
1. جمع أخبار عربية
2. تعريف Schema للبيانات المطلوبة
3. استخدام GPT/Llama كـ Teacher
5. إنشاء SFT Dataset
6. تجهيز Dataset لـ LLaMA-Factory
7. عمل LoRA Fine-Tuning على Qwen
8. تقييم النموذج بعد التدريب
9. نشر النموذج باستخدام vLLM
10. عمل Load Testing وقياس الأداء


## Setup

In [1]:
#!pip install -qU torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0
!pip install -qU json-repair==0.29.1
!pip install -qU groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
!pip install -qU vllm==0.7.2 locust faker==35.2.0 wandb

## Mount Google Drive

In [2]:
try:
    from google.colab import drive
except ImportError:
    print("Not running in Google Colab")
drive.mount('/gdrive')

Mounted at /gdrive


###LLaMA-Factory تنزيل مشروع

In [3]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git -q
!cd LLaMA-Factory && pip install -e . -q
# --depth 1 <<<<< اخرCommit

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!llamafactory-cli version

----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.6.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------


## Imports

In [5]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from openai import OpenAI
from groq import Groq

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair
import wandb
from google.colab import userdata

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from huggingface_hub import login

data_dir = "/gdrive/MyDrive/f-t-f/data"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
groq_model_id = "llama-3.3-70b-versatile"

device = "cuda" if torch.cuda.is_available() else "cpu"

torch_dtype = None

def parse_json(text):  ## JSON صلاح أخطاء
    try:
        return json_repair.loads(text)
    except:
        return None

In [6]:
torch.__version__

'2.11.0+cpu'

###Read keys

In [7]:
#wandb_key = userdata.get("wandb")
hf_token = userdata.get("huggingface")

if not hf_token:
    raise ValueError("huggingface secret not found")

"""
if not wandb_key:
    raise ValueError("wandb secret not found")

wandb.login(key=wandb_key)
login(token=hf_token)"""

'\nif not wandb_key:\n    raise ValueError("wandb secret not found")\n\nwandb.login(key=wandb_key)\nlogin(token=hf_token)'

###GROQ_API_KEY

In [8]:
GROQ_API_KEY = userdata.get("groq")

if not GROQ_API_KEY:
    raise ValueError("GROQ API KEY not found in Colab Secrets")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print(" GROQ API KEY Loaded Successfully Tmaam")

 GROQ API KEY Loaded Successfully Tmaam


## Tasks
الخبر ده مجرد Sample للتجربة

الغرض منه مش التدريب

الغرض منه اختبار النظام
استخراج معلومات منظمة من الخبر

In [9]:
story = """
ذكرت مجلة فوربس أن العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال،
 حيث تتأثر هذه العلاقة بأنماط السلوك المالي المتوارثة عبر الأجيال.

التقرير الذي يستند إلى أبحاث الأستاذ الجامعي شاين إنيت حول
الرفاه المالي يوضح أن لكل شخص "شخصية مالية" تتحدد وفقا لطريقة
 تفاعله مع المال، والتي تتأثر بشكل مباشر بتربية الأسرة وتجارب الطفولة.

 الأبعاد الثلاثة للعلاقة بالمال
بحسب الدراسة، هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال:

الاكتساب (A): يميل الأفراد الذين ينتمون لهذا
 البعد إلى اعتبار المال سلعة قابلة للجمع، حيث يرون
في تحقيق الثروة هدفا بحد ذاته. والجانب السلبي لهذا
 النمط هو إمكانية التحول إلى هوس بالثروة أو العكس،
 أي رفض تام لاكتساب المال باعتباره مصدرا للفساد.

الاستخدام (U): يرى هؤلاء الأشخاص المال أداة للتمتع بالحياة، حيث يربطون قيمته بقدرته على توفير
المتعة والراحة. ومع ذلك، قد يصبح
البعض مدمنا على الإنفاق، في حين يتجه آخرون إلى التقشف المفرط خوفا من المستقبل.

الإدارة (M): أصحاب هذا النمط يعتبرون المال مسؤولية تتطلب التخطيط الدقيق. لكن في بعض الحالات،
 قد يتحول الأمر إلى هوس مفرط بإدارة الإنفاق، مما يؤثر سلبا على العلاقات الشخصية.

 كيف تؤثر العائلة على علاقتنا بالمال؟
يشير التقرير إلى أن التجارب الأسرية تلعب دورا رئيسيا في تحديد
 "الشخصية المالية" لكل فرد، على سبيل المثال، إذا كان أحد الوالدين يعتمد على المال
كمكافأة للسلوك الجيد، فقد يتبنى الطفل لاحقا النمط نفسه في حياته البالغة.

لتحليل هذه التأثيرات بشكل دقيق، طورت رابطة العلاج المالي
(Financial Therapy Association) أداة تسمى مخطط الجينوم المالي (Money Genogram)،
وهو نموذج يُستخدم لتحديد الأنماط المالية داخل العائلة.

تتضمن هذه الأداة:

رسم شجرة عائلية.
تصنيف أفراد العائلة وفقا للأبعاد الثلاثة للعلاقة بالمال (A ،U ،M).
تحديد ما إذا كان السلوك المالي لكل فرد صحيا (+) أو غير صحي (-).
على سبيل المثال، إذا نشأ شخص في عائلة
اعتادت على الإنفاق المفرط، فقد يكون لديه ميل قوي إلى اتباع النمط نفسه،
 أو العكس تماما، حيث يصبح مقتصدا بشكل مبالغ فيه كرد فعل نفسي.
"""

### Details Extraction
**Structured Output Schema for Text Extraction System**
   ### Pydantic Schema
*title      
*summary            
*entities               
*keywords            
*category

### أنا عايز الناتج بالشكل ده بالظبط.

In [10]:
"""
عاوز الاوتبت يكون كدا
{
   title:
   category:
   entities:
   summary:
   keywords:
}

"""

'\nعاوز الاوتبت يكون كدا\n{\n   title:\n   category:\n   entities:\n   summary:\n   keywords:\n}\n\n'

In [11]:
StoryCategory = Literal[  #  تصنيف النص
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"]

EntityType = Literal[  # نواع الكيانات
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"]

class Entity(BaseModel):   # نوع الكيان من القائمة المحددة
    entity_value: str = Field(..., description="The actual name or value of the entity.")   # الاسم الحقيقي للكيان داخل النص
    entity_type: EntityType = Field(..., description="The type of recognized entity.")    # نوع الكيان داخل القايمه

class NewsDetails(BaseModel):   # عنوان احترافي
    story_title: str = Field(..., min_length=5, max_length=300,
                             description="A fully informative and SEO optimized title of the story.")

    story_keywords: List[str] = Field(..., min_items=1,  # استخراج كلمات مفتاحية
                                       description="Relevant keywords associated with the story.")

    story_summary: List[str] = Field( # تلخيص النص إلى نقاط
                                    ..., min_items=1, max_items=5,
                                    description="Summarized key points about the story (1-5 points).")

    story_category: StoryCategory = Field(..., description="Category of the news story.")  # تصنيف المقال إلى category واحدة فقط

    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")
                   # استخراج كل الكيانات داخل النص


/tmp/ipykernel_1183/3379463452.py:18: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1,  # استخراج كلمات مفتاحية
/tmp/ipykernel_1183/3379463452.py:21: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field( # تلخيص النص إلى نقاط
/tmp/ipykernel_1183/3379463452.py:21: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field( # تلخيص النص إلى نقاط
/

###Chat Prompt Template

 NLP Parser أنت

وده الخبر

Schema وده ال

رجع JSON فقط

In [12]:
details_extraction_messages = [{
        "role": "system",
        "content": "\n".join([
            "You are an NLP data paraser.",
            "You will be provided by an Arabic text associated with a Pydantic scheme.",
            "Generate the ouptut in the same story language.",
            "You have to extract JSON details from text according the Pydantic details.", # schema يربط النموذج ب
            "Extract details as mentioned in text.",
            "Do not generate any introduction or conclusion."])},

    {"role": "user",
        "content": "\n".join([
            "## Story:",
            story.strip(),
            "",
            "## Pydantic Details:",
            json.dumps(
                NewsDetails.model_json_schema(), ensure_ascii=False),
            "",
            "## Story Details:",  # ابدأ output في شكل JSON
            "```json"])}]

### Translation

In [13]:
class TranslatedStory(BaseModel):
    translated_title: str = Field(..., min_length=5, max_length=300,
                                  description="Suggested translated title of the news story.")
    translated_content: str = Field(..., min_length=5,
                                    description="Translated content of the news story.")

targeted_lang = "English"

translation_messages = [{
        "role": "system",
        "content": "\n".join([
            "You are a professional translator.",
            "You will be provided by an Arabic text.",
            "You have to translate the text into the `Targeted Language`.",
            "Follow the provided Scheme to generate a JSON",
            "Do not generate any introduction or conclusion."])},
    {"role": "user",
        "content":  "\n".join([
            "## Story:",
            story.strip(),
            "",

            "## Pydantic Details:",
            json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
            "",

            "## Targeted Language:",
            targeted_lang,
            "",

            "## Translated Story:",
            "```json"])}]

## Baseline Evaluation

In [14]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [15]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

###LLM Inference Pipeline Details_extraction

###   هنا بيختبر النموذج الأساسي
         هل بيعرف يترجم Extraction وبيشوف هل بيعرف يعمل Fine-Tuning قبل أي




In [16]:
text = tokenizer.apply_chat_template(
    details_extraction_messages,
    tokenize=False,
    add_generation_prompt=True)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,)

generated_ids = [  # إزالة  Prompt من الناتج
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

response = tokenizer.batch_decode(generated_ids, # Decoding
                                  skip_special_tokens=True)[0]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


###LLM Inference Pipeline Machine translation

In [17]:
text = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

response_t = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [18]:
print(response)

{
  "story_title": "How Family Shapes Our Financial Relationships",
  "story_keywords": [
    "family influence on money",
    "financial relationships",
    "inheritance patterns",
    "moneymaking habits",
    "personal finance"
  ],
  "story_summary": [
    "Families play a crucial role in shaping how individuals interact with money.",
    "This relationship is influenced by inherited behaviors across generations."
  ],
  "story_category": "economy",
  "story_entities": [
    {
      "entity_value": "Forbes Magazine",
      "entity_type": "organization"
    },
    {
      "entity_value": "Shane Einot",
      "entity_type": "person-female"
    },
    {
      "entity_value": "Financial Therapy Association",
      "entity_type": "organization"
    }
  ]
}


In [19]:
print(response_t)

{
  "translated_title": "Forbes Magazine Reveals Family Plays a Central Role in Forming Individuals' Financial Relationships",
  "translated_content": "According to Forbes magazine, family plays a crucial role in shaping individuals' financial relationships, as these relationships are influenced by inherited behavioral patterns across generations."
}


## Evaluate OpenAI
مقارنة مع نماذج أقوى
علشان يستخدمهم Teacher Models

In [21]:
openai_client = OpenAI(
    api_key=userdata.get("oai"))
    # base_url="http://localhost:8000"

openai_model_id = "gpt-4o-mini"

###Evaluate Groq

In [22]:
groq_client = Groq(
    api_key=userdata.get("groq"))

groq_model_id = "llama-3.3-70b-versatile"

### LLM_Openai details_extraction

In [ ]:
chat_completion = openai_client.chat.completions.create(
    messages=details_extraction_messages,
    model=openai_model_id,
    temperature=0.2,
    response_format={"type": "json_object"})

print(chat_completion.choices[0].message.content)

###LLM_Openai translation

In [ ]:
chat_completion = openai_client.chat.completions.create(
    messages=translation_messages,
    model=openai_model_id,
    temperature=0.2,
    response_format={"type": "json_object"})

print(chat_completion.choices[0].message.content)

### LLM_Groq details_extraction
### يرسل الرسائل إلى نموذج Groq ويطلب منه إرجاع النتيجة بصيغة JSON، ثم يطبع الناتج.

In [23]:
chat_completion = groq_client.chat.completions.create(
    messages=details_extraction_messages,
    model=groq_model_id,
    temperature=0.2,
    response_format={"type": "json_object"}) #بيشيل اول علامه في جيسون فايل

print(chat_completion.choices[0].message.content)

{
  "story_title": "العلاقة بين الأسرة والمال",
   "story_keywords": [
      "المال",
      "العائلة",
      "التجارب الأسرية",
      "الشخصية المالية",
      "الرفاه المالي"
   ],
   "story_summary": [
      "العلاقة بين الأسرة والمال تعتمد على أنماط السلوك المالي المتوارثة عبر الأجيال",
      "الشخصية المالية تتأثر بتربية الأسرة وتجارب الطفولة",
      "هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال: الاكتساب، الاستخدام، والإدارة",
      "التجارب الأسرية تلعب دورا رئيسيا في تحديد الشخصية المالية لكل فرد",
      "مخطط الجينوم المالي هو أداة لتحديد الأنماط المالية داخل العائلة"
   ],
   "story_category": "economy",
   "story_entities": [
      {
         "entity_value": "فوربس",
         "entity_type": "organization"
      },
      {
         "entity_value": "شاين إنيت",
         "entity_type": "person-male"
      },
      {
         "entity_value": "الرفاه المالي",
         "entity_type": "not_specified"
      },
      {
         "entity_value": "رابطة العلاج المالي",
         "entity_ty

###LLM_Groq translation

In [24]:
chat_completion_t = groq_client.chat.completions.create(
    messages=translation_messages,
    model=groq_model_id,
    temperature=0.2,
    response_format={"type": "json_object"}) #بيشيل اول علامه في جيسون فايل

print(chat_completion_t.choices[0].message.content)

{
  "translated_title": "How Family Influences Our Relationship with Money",
   "translated_content": "Forbes magazine noted that family plays a central role in shaping individuals' relationship with money, as this relationship is affected by inherited financial behavior patterns across generations. A report based on Professor Shane Enete's research on financial well-being explains that each person has a 'financial personality' determined by how they interact with money, which is directly influenced by family upbringing and childhood experiences. According to the study, there are three main dimensions that shape our relationship with money: Acquisition (A), where individuals tend to consider money as a collectible commodity and see wealth accumulation as a goal in itself, Usage (U), where people view money as a tool for enjoying life and associate its value with the ability to provide pleasure and comfort, and Management (M), where individuals consider money a responsibility that requi

In [25]:
print(chat_completion.choices[0].message.content)

{
  "story_title": "العلاقة بين الأسرة والمال",
   "story_keywords": [
      "المال",
      "العائلة",
      "التجارب الأسرية",
      "الشخصية المالية",
      "الرفاه المالي"
   ],
   "story_summary": [
      "العلاقة بين الأسرة والمال تعتمد على أنماط السلوك المالي المتوارثة عبر الأجيال",
      "الشخصية المالية تتأثر بتربية الأسرة وتجارب الطفولة",
      "هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال: الاكتساب، الاستخدام، والإدارة",
      "التجارب الأسرية تلعب دورا رئيسيا في تحديد الشخصية المالية لكل فرد",
      "مخطط الجينوم المالي هو أداة لتحديد الأنماط المالية داخل العائلة"
   ],
   "story_category": "economy",
   "story_entities": [
      {
         "entity_value": "فوربس",
         "entity_type": "organization"
      },
      {
         "entity_value": "شاين إنيت",
         "entity_type": "person-male"
      },
      {
         "entity_value": "الرفاه المالي",
         "entity_type": "not_specified"
      },
      {
         "entity_value": "رابطة العلاج المالي",
         "entity_ty

In [26]:
json.loads(chat_completion.choices[0].message.content)

{'story_title': 'العلاقة بين الأسرة والمال',
 'story_keywords': ['المال',
  'العائلة',
  'التجارب الأسرية',
  'الشخصية المالية',
  'الرفاه المالي'],
 'story_summary': ['العلاقة بين الأسرة والمال تعتمد على أنماط السلوك المالي المتوارثة عبر الأجيال',
  'الشخصية المالية تتأثر بتربية الأسرة وتجارب الطفولة',
  'هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال: الاكتساب، الاستخدام، والإدارة',
  'التجارب الأسرية تلعب دورا رئيسيا في تحديد الشخصية المالية لكل فرد',
  'مخطط الجينوم المالي هو أداة لتحديد الأنماط المالية داخل العائلة'],
 'story_category': 'economy',
 'story_entities': [{'entity_value': 'فوربس', 'entity_type': 'organization'},
  {'entity_value': 'شاين إنيت', 'entity_type': 'person-male'},
  {'entity_value': 'الرفاه المالي', 'entity_type': 'not_specified'},
  {'entity_value': 'رابطة العلاج المالي', 'entity_type': 'organization'}]}

## Knowledge Distillation
## JSONL تحميل الداتا من ملف


In [27]:
data_dir

'/gdrive/MyDrive/f-t-f/data'

###Load datasets from news-sample.jsonl

In [28]:
raw_data_path = join(data_dir, "datasets", "news-sample.jsonl")

raw_data = []
for line in open(raw_data_path,"r", encoding="utf-8"):
    if line.strip() == "":
        continue

    raw_data.append(
        json.loads(line.strip()))

random.Random(101).shuffle(raw_data)

print(f"Raw data: {len(raw_data)}")

Raw data: 2400


In [29]:
raw_data[5]['content']

'بدأ منتجع ياباني معروف بشوارعه المغطاة بالثلوج في تقييد وصول الزوار اليوم الاثنين، في محاولة لمكافحة السياحة المفرطة خلال فصل الشتاء. \n يجذب منتجع غينزان أونسن، وهي بلدة معزولة نسبيا تقع في مقاطعة ياماغاتا في شمال شرق اليابان، نحو 330 ألف زائر كل عام. \n ويشارك كثر منهم صورا على شبكات التواصل الاجتماعي لمبانيها القديمة المغطاة بالثلوج، والتي تضاء عند حلول الليل بالوهج البرتقالي لمصابيح الشوارع، مما يخلق جوا سحريا. \n لكن سلطات منتجع غينزان أونسن، مثل تلك الموجودة في الوجهات الأكثر شهرة مثل كيوتو أو جبل فوجي، قررت اتخاذ تدابير في مواجهة المشاكل المرورية المتزايدة والمشاجرات وغيرها من المضايقات المرتبطة بهذا التدفق. \n فاعتبارا من اليوم الاثنين، سيُسمح فقط للأشخاص الذين ينزلون في الفنادق المحلية بالدخول إلى المنطقة بعد الساعة الثامنة مساءً، لكن سيحتاج أولئك الذين يرغبون في الزيارة بين الساعة 1700 و2000 إلى الحجز مسبقا. \n وشهدت اليابان تدفقا قياسيا من السياح الأجانب هذا العام، بدفع من تراجع قيمة الين الذي أسهم بشكل ملحوظ في ازدياد معدلات السفر بعد الجائحة. \n وقال تاكايوكي سايتو، رئيس 

###SFT  > Annotation يعمل
###
      Chatbot مش  Annotators بيشتغل Llama

###أخبار Dataset عنده
 Llama لكل خبر يرسله        
 ويقول
 استخرج البيانات المنظمة
   

In [30]:
raw_data_test = raw_data[:10]

In [31]:
cloud_model_id = groq_model_id
price_per_1m_input_tokens = 0.150   # حساب تكلفة التشغيل
price_per_1m_output_tokens = 0.600

prompt_tokens = 0
completion_tokens = 0

save_to = join(data_dir, "datasets", "sft101.jsonl")

ix = 0
for story in tqdm(raw_data_test):
    sample_details_extraction_messages = [
        {
            "role": "system",   # تحديد سلوك النموذج
            "content": "\n".join([
                "You are an NLP data paraser.",
                "You will be provided by an Arabic text associated with a Pydantic scheme.",
                "Generate the ouptut in the same story language.",
                "You have to extract JSON details from text according the Pydantic details.",
                "Extract details as mentioned in text.",
                "Do not generate any introduction or conclusion."
            ])},
        {
            "role": "user",
            "content": "\n".join([
                "## Story:",
                story['content'].strip(),
                "",

                "## Pydantic Details:",
                json.dumps(
                    NewsDetails.model_json_schema(), ensure_ascii=False
                ),
                "",
                "## Story Details:",
                "```json"
            ])}]

    response = groq_client.chat.completions.create(
                            messages=sample_details_extraction_messages,
                            model=cloud_model_id,
                            temperature=0.2,)

    if response.choices[0].finish_reason != "stop":
        prompt_tokens += response.usage.prompt_tokens
        continue

    llm_response = response.choices[0].message.content
    llm_resp_dict = parse_json(llm_response)

    if not llm_resp_dict:
        continue

    with open(save_to, "a", encoding="utf8") as dest:
        dest.write(json.dumps({
            "id": ix,
            "story": story['content'].strip(),
            "task": "Extrat the story details into a JSON.",
            "output_scheme": json.dumps( NewsDetails.model_json_schema(), ensure_ascii=False ),
            "response": llm_resp_dict,
        }, ensure_ascii=False, default=str)  + "\n" )

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens

    if(ix % 3) == 0:
        cost_input = (prompt_tokens / 1_000_000) * price_per_1m_input_tokens
        cost_output = (completion_tokens / 1_000_000) * price_per_1m_output_tokens
        total_cost = cost_input + cost_output

        print(f"Iteration {ix}: Total Cost = ${total_cost:.4f} ")

  0%|          | 0/10 [00:00<?, ?it/s]

Iteration 3: Total Cost = $0.0014 
Iteration 6: Total Cost = $0.0026 
Iteration 9: Total Cost = $0.0041 


In [32]:
cloud_model_id = groq_model_id

price_per_1m_input_tokens = 0.150
price_per_1m_output_tokens = 0.600

prompt_tokens = 0
completion_tokens = 0

save_to = join(data_dir, "datasets", "sft101_translate.jsonl")
ix = 0
for story in tqdm(raw_data_test):

    for targeted_lang in ["English", "French"]:
        sample_translation_messages = [
            {
                "role": "system",
                "content": "\n".join([
                    "You are a professional translator.",
                    "You will be provided by an Arabic text.",
                    "You have to translate the text into the `Targeted Language`.",
                    "Follow the provided Scheme to generate a JSON",
                    "Do not generate any introduction or conclusion."
                ])},
            {
                "role": "user",
                "content": "\n".join([
                    "## Pydantic Details:",
                    json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
                    "",

                    "## Targeted Language or Dialect:",
                    targeted_lang,
                    "",

                    "## Story:",
                    story['content'].strip(),
                    "",

                    "## Translated Story:",
                    "```json"
                ])}]

        response = groq_client.chat.completions.create(
                            messages=sample_details_extraction_messages,
                            model=cloud_model_id,
                            temperature=0.2,
                        )

        if response.choices[0].finish_reason != "stop":
            prompt_tokens += response.usage.prompt_tokens
            continue

        llm_response = response.choices[0].message.content
        llm_resp_dict = parse_json(llm_response)

        if not llm_resp_dict:
            continue

        with open(save_to, "a", encoding="utf8") as dest:
            dest.write(json.dumps({
                "id": ix,
                "story": story['content'].strip(),

                "output_scheme": json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
                "task": f"You have to translate the story content into {targeted_lang} associated with a title into a JSON.",

                "response": llm_resp_dict, }
                , ensure_ascii=False, default=str)  + "\n" )

        ix += 1
        prompt_tokens += response.usage.prompt_tokens
        completion_tokens += response.usage.completion_tokens

        if(ix % 3) == 0:
            cost_input = (prompt_tokens / 1_000_000) * price_per_1m_input_tokens
            cost_output = (completion_tokens / 1_000_000) * price_per_1m_output_tokens
            total_cost = cost_input + cost_output

            print(f"Iteration {ix}: Total Cost = ${total_cost:.4f} ")

  0%|          | 0/10 [00:00<?, ?it/s]

Iteration 3: Total Cost = $0.0015 
Iteration 6: Total Cost = $0.0029 
Iteration 9: Total Cost = $0.0043 
Iteration 12: Total Cost = $0.0057 
Iteration 15: Total Cost = $0.0071 


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqnxv8zffypatfeh0rw2hze2` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98830, Requested 2130. Please try again in 13m49.44s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Format Finetuning Dataset Generation
###  LLaMA-Factory محتاج Format معين

###التدريب  Dataset  يبني

In [33]:
sft_data_path = join(data_dir, "datasets", "sft101.jsonl")
llm_finetunning_data = []  #container

system_message = "\n".join([
    "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."])

for line in open(sft_data_path):
    if line.strip() == "":
        continue

    rec = json.loads(line.strip())

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

random.Random(101).shuffle(llm_finetunning_data)

In [34]:
len(llm_finetunning_data)

10

### Train ,Validation يقسم البيانات إلى

In [35]:
int(len(llm_finetunning_data)*(2/3))
# هنا بخلي الترين سايز ثلثي الداتا

6

In [36]:
train_sample_sz = int(len(llm_finetunning_data) * (2 / 3))
train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

os.makedirs(join(data_dir, "datasets"), exist_ok=True)

with open(join(data_dir, "datasets", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "datasets", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [37]:
join(data_dir, "datasets", "val.json")

'/gdrive/MyDrive/f-t-f/data/datasets/val.json'

## Finetuning

In [ ]:
# # Configure LLaMA-Factory for the new datasets

# # update /content/LLaMA-Factory/data/dataset_info.json and append
# ```
   "news_finetune_train": {
        "file_name": "/gdrive/MyDrive/f-t-f/data/datasets/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/gdrive/MyDrive/f-t-f/data/datasets/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
# ```

# https://wandb.ai/mr-bakrianoo/llamafactory/runs/apwbkni9
# https://wandb.ai/mr-bakrianoo/llamafactory/runs/c5tf0q90

In [38]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/f-t-f/data/datasets/checkpoint-1500
output_dir: /gdrive/MyDrive/f-t-f/data/datasets/
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: false
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

# report_to: wandb
run_name: newsx-finetune-llamafactory

# push_to_hub: true
# export_hub_model_id: "ismail/news-analyzer"
# hub_private_repo: true
# hub_strategy: checkpoint


Writing /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


### هعمل هنا ترين
Adapter واحمل ال


In [39]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-06-26 14:57:41] llamafactory.hparams.parser:149 >> We recommend enable mixed precision training.
[INFO|2026-06-26 14:57:41] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cpu, distributed training: False, compute dtype: None
[INFO|configuration_utils.py:771] 2026-06-26 14:57:41,982 >> loading configuration file config.json from cache at /root/.cache/h

## New Finetuned Model Evaluation

In [ ]:
base_model_id

'Qwen/Qwen2.5-1.5B-Instruct'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

In [ ]:
finetuned_model_id = "/gdrive/MyDrive/f-t-f/data/datasets"
model.load_adapter(finetuned_model_id)

ValueError: adapter model file not found in /gdrive/MyDrive/f-t-f/data/datasets. Make sure you are passing the correct path to the adapter model.

### هل النموذج المتخصص أصبح أفضل

In [ ]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True)

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,)

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(translation_messages)

In [ ]:
parse_json(response)

{'translated_title': "Forbes Magazine Reveals Family Plays a Central Role in Forming Individuals' Financial Relationships",
 'translated_content': "According to Forbes magazine, family plays a crucial role in shaping individuals' financial relationships, as these relationships are influenced by inherited behavioral patterns across generations."}

#### Tip for Qwen2.5
### Qwen أحيانًا يولد رموز صينية
### دي خطوة Post-Processing وتحسين جودة  Output
###


Qwen2.5 oftenly produce chinese characters with some responses. To skip this, use the next class to generate responses.

Source:
`https://jupyter267.medium.com/how-to-eliminate-the-chance-of-generating-chinese-in-qwen-2-5-2cf919bb0fdc`



In [ ]:
class Generator:
    def __init__(self, model, tokenizer):

        self.model, self.tokenizer = model, tokenizer
        self.mask = None

    def generate(self, messages:list, max_new_tokens: int=2000, temperature:float=0.1):

        def logits_processor(token_ids, logits):
          # logits_processor default recieve the logits which is the score matrix of each time-step
          """
              A processor to ban Chinese character
          """
          if self.mask is None:
              # as we don't know where the Chinses tokens locate at which index
              # in the vocabulary but we know how it looks like and the range of it

              # decode all the tokens in the vocabulary in order
              token_ids = torch.arange(logits.size(-1))
              decoded_tokens = self.tokenizer.batch_decode(token_ids.unsqueeze(1), skip_special_tokens=True)

              # create a mask tensor to exclude positions of Chinese characters.
              # since this process uses a for loop and is time-consuming,
              # the result will be stored as a property for later use to ensure it only runs once.
              self.mask = torch.tensor([
                  # loop through each token in the vocabulary and compare it to Chinese characters.
                  any(0x4E00 <= ord(c) <= 0x9FFF or 0x3400 <= ord(c) <= 0x4DBF or 0xF900 <= ord(c) <= 0xFAFF for c in
                      token)
                  for token in decoded_tokens
              ])

          # mask the score by - inf
          logits[:, self.mask] = -float("inf")
          return logits

        # this step transforms the messages into a string,
        # adding special tokens e.g separate tokens between system content user queries
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            # add the logits_processor here
            logits_processor=[logits_processor]
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

In [ ]:
# define an object
llm = Generator(model, tokenizer)

# generate a response without chinese characters
response = llm.generate(details_extraction_messages)
print( parse_json(response) )

response = llm.generate(translation_messages)
print( parse_json(response) )

## Cost Estimation

###عدد  Tokens الزمن و التكلفة



In [ ]:
from tqdm.auto import tqdm
from faker import Faker
import random
from datetime import datetime

start_time = datetime.now()
fake = Faker('ar')

input_tokens = 0
output_tokens = 0

for i in tqdm(range(30)):
    prompt = fake.text(max_nb_chars=random.randint(150, 200))

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    response = generate_resp(messages)

    input_tokens += len(tokenizer.apply_chat_template(messages))
    output_tokens += len(tokenizer.encode(response))

total_time = (datetime.now() - start_time).total_seconds()

print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

  0%|          | 0/30 [00:00<?, ?it/s]

Total Time: 149.491832 seconds
Input Tokens: 2377
Output Tokens: 2157
Total Tokens: 4534


In [ ]:
# Total Time: 387.838115 seconds
# Input Tokens: 2446
# Output Tokens: 7375
# Total Tokens: 9821

In [ ]:
9821  / 388

25.311855670103093

## vLLM

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_model_id = "/gdrive/MyDrive/f-t-f/data/datasets"


!nohup vllm serve "{base_model_id}" --dtype=half --gpu-memory-utilization 0.8 --max_lora_rank 64 --enable-lora --lora-modules news-lora="{adapter_model_id}" &


nohup: appending output to 'nohup.out'


In [ ]:
!tail -n 30 nohup.out

tail: cannot open 'nohup.out' for reading: No such file or directory


### API Inference
### SaaS

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

prompt = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True)

In [ ]:
vllm_model_id = "news-lora"

llm_response = requests.post("http://localhost:8000/v1/completions", json={
    "model": vllm_model_id,
    "prompt": prompt,
    "max_tokens": 1000,
    "temperature": 0.3
})

llm_response.json()

## Load Testing هنعمل ستريس تيست علي الموديل
###  يعمل Users وهميين  
### في نفس الوقت 20 مستخدم

In [ ]:
%%writefile locust.py

import random
import json
from locust import HttpUser, task, between, constant
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "news-lora"
        prompt = fake.text(max_nb_chars=random.randint(150, 200))

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.3}

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")


Overwriting locust.py


In [ ]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

In [ ]:
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")

Total Input Tokens: 2662
Total Output Tokens: 37840


In [ ]:
37840 / 60

630.6666666666666